# ML Assignment 2 — BITS Virtual Lab run

Trains, tunes and evaluates the five classifiers on the UCI *Online Shoppers Purchasing Intention* dataset, using the same code as `train.py`.

**Run on BITS Virtual Lab:** upload this repository, open this notebook from the repository folder, then *Kernel → Restart Kernel and Run All Cells*. Takes about a minute.

## 1. Environment (proof of where this ran)

In [ ]:
import datetime, platform, socket, sys
import numpy as np, pandas as pd, sklearn

print("Host     :", socket.gethostname())
print("Platform :", platform.platform())
print("Python   :", sys.version.split()[0])
print("Packages : scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
print("Run at   :", datetime.datetime.now().strftime("%d %b %Y, %H:%M:%S"))

## 2. Load the data and make the stratified 80/20 split

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S", force=True)

import train
from shopintent.schema import FEATURES, TARGET

df = train.load_dataset()
train_df, test_df = train.split_dataset(df)
print(f"{len(FEATURES)} features | train {len(train_df):,} rows | test {len(test_df):,} rows")
df.head()

In [ ]:
pd.DataFrame({
    "train": train_df[TARGET].value_counts(normalize=True).round(4),
    "test": test_df[TARGET].value_counts(normalize=True).round(4),
}).rename_axis("Revenue (purchase?)")

## 3. Tune (5-fold CV on MCC) and train the five classifiers

In [ ]:
models = train.train_all(train_df)
pd.DataFrame({m.spec.name: {"best params": m.best_params, "CV MCC": round(m.cv_mcc, 4), "seconds": round(m.seconds, 1)} for m in models}).T

## 4. Evaluate on the 2,441-row hold-out set — the six required metrics

In [ ]:
table = train.score_all(models, test_df)
table.round(4)

## 5. Confusion matrices  `[[TN, FP], [FN, TP]]`

In [ ]:
from shopintent.evaluation import evaluate
from shopintent.schema import prepare_features, prepare_target

X_test, y_test = prepare_features(test_df), prepare_target(test_df)
for m in models:
    cm = evaluate(m.pipeline, X_test, y_test).confusion
    print(f"{m.spec.name:24s} TN={cm[0,0]:5d}  FP={cm[0,1]:4d}  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}")

## 6. Save (optional)

The models committed to GitHub were saved with the versions pinned in `requirements.txt`, which the Streamlit app needs. Leave `SAVE = False` here so a different scikit-learn on the lab cannot produce incompatible model files.

In [ ]:
SAVE = False
if SAVE:
    train.save_artifacts(models, table, test_df)
print("Done.")